# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides an interactive walkthrough for programmatically loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will let us inspect the dataset structure, including record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata: name, description
meta = dataset.metadata

print(f"Dataset: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and field `@id`s, to understand the dataset's table-like structure before extracting any records.

We use the metadata interface to access record sets and their fields.

In [ ]:
# Discover record sets and their fields via their @id
record_sets = dataset.metadata.record_sets

print("Record sets and fields in the dataset:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}   @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name}   @id: {field.id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis.

Use the record set and field `@id`s from the overview above to reference the data structures unambiguously. Here we will load *all* the record sets into Python variables dynamically.

In [ ]:
# Prepare a dictionary of DataFrames by RecordSet @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# List columns for each record set
for rs_id, df in dataframes.items():
    print(f"RecordSet @id: {rs_id}")
    print("Columns (field @id):", df.columns.tolist())
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)

Let's select a RecordSet and perform EDA tasks, such as filtering, normalization, and grouping. All fields are referenced by their `@id` (*not* names).

You should adapt the following cell by picking a RecordSet from those found above. For this demo, we pick the main survey responses RecordSet (replace with the correct one found above if needed), and select a numeric field, e.g., `phq9_score` (use the actual `@id` displayed in the previous step).

In [ ]:
# --- Parameters to adjust based on IDs printed above ---
# Choose the main survey responses RecordSet and its key numeric field, by @id.
# For example, suppose record_set_id looks like 'https://api.app.sen.science/frontiers/7160411/RESPONSE-RECORDSET-ID'
# Suppose the GAD-7 score column is '@gad7_score', and village is '@village'

# Please adapt these IDs below based on printed output above!
main_record_set_id = record_set_ids[0]  # Change index or id as appropriate
df = dataframes[main_record_set_id]
numeric_field_id = [col for col in df.columns if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()]
if numeric_field_id:
    numeric_field_id = numeric_field_id[0]
else:
    numeric_field_id = df.columns[0]  # fallback

# Print the numeric field we'll analyze
print(f"Numeric field selected: {numeric_field_id}")

# Drop missing values for simplicity
df_numeric = df.dropna(subset=[numeric_field_id])

# Filtering data: e.g., only keep records with scores above a threshold
threshold = 10
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized" ]].head())

# Group by a categorical key, e.g., village or gender
possible_group_fields = [col for col in df.columns if 'village' in col.lower() or 'gender' in col.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib or seaborn. Here, we plot the distribution of a selected score (e.g., GAD-7), and optionally its distribution by group (such as village or gender, if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df_numeric[numeric_field_id], kde=True, color='teal')
plt.title(f"Distribution of {numeric_field_id} scores")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If a group field is present, show boxplot by group
if group_field is not None:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df_numeric[group_field], y=df_numeric[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Programmatically access a Croissant dataset using the `mlcroissant` library,
- Discover and reference all record sets and fields using their `@id`s,
- Load the tabular survey data into `pandas` DataFrames,
- Perform basic EDA, including filtering, normalization, grouping, and visualization.

This workflow enables reproducible, scalable analysis for data described in the Croissant standard. For further exploration, consider deeper statistical modeling, feature engineering, or integrating with downstream machine learning pipelines.